# TripMe - Sri Lankan Food Classifier (Kaggle version: fine-tune on 4 merged Roboflow datasets)

Merges 4 Roboflow object-detection exports, uploaded as a single Kaggle
Dataset (zip of the local `data/raw_datasets/` folder):

1. `sri_lankan_food_detect_v6_coco` - 795 images (307 base + Roboflow
   augmentation), 11 main-dish classes (Dhal Curry, Fish Curry, Gotukola
   Mallum, Hoppers, Kiribath, Kottu, Lunu sambol, Pittu, Pol Sambol, String
   Hoppers, Watalappam).
2. `String Hoppers.v1i.coco` - 372 images, adds a large boost specifically
   to the string_hoppers class.
3. `Sri Lankan Dishes.v2i.coco` - 53 images, dhal_curry/fish_curry/milk_rice.
4. `Automatic Billing System for Sri Lankan Pastry Shop...v1i.coco` - 52
   images, 8 bakery/pastry classes (Donut, Egg roll, Fish Bun, Pastry,
   Patis, Sandwich, Sausage Bun, Spanchi) - different scope (bakery items,
   not main dishes), merged in anyway to widen recognition coverage.

All 4 are bounding-box annotated, not classification folder sets, so this
notebook:

1. Auto-detects the uploaded dataset folder under `/kaggle/input`.
2. Crops each bounding box out of its source image - each crop becomes one
   classification training example, labeled with that box's class. Class
   names are normalized (lowercased, spaces/hyphens to underscores) so the
   same dish name from different sources lands in one folder, not two, and
   each source's root/supercategory placeholder class (e.g. "food",
   "lankan") is skipped rather than trained on.
3. Fine-tunes a small pretrained vision model (`google/vit-base-patch16-224-in21k`)
   on the merged crops - well under 1,300 images total, so on a T4 GPU
   this normally finishes in well under 30 minutes.
4. Saves the fine-tuned model and zips it - click **Save Version** after
   the run finishes, then download the zip from the **Output** tab (Kaggle
   has no direct browser-auto-download, unlike Colab's files.download()).

## Honest caveat
Even merged, this is roughly 1,000-1,300 images across ~20+ classes (many
classes still only have a few dozen examples, and the pastry classes are
especially thin at ~5-6 images each). This fine-tune will very likely
overfit and should be treated as a first pass, not a production-grade
classifier - the CLIP zero-shot approach in `serve/food_scan.py` remains
the more robust default for any dish outside what this model was actually
trained on.

## How to run this on Kaggle
1. Zip your local `data/raw_datasets/` folder (contains the 4 dataset
   subfolders) and upload it as a Kaggle Dataset (e.g. named
   `tripme-food-datasets`), then attach it to this notebook via "Add Input".
2. Turn on a GPU accelerator (Settings -> Accelerator -> GPU T4 x2 or better).
3. Run all cells. Every step prints what it's doing and how many
   images/crops/classes it found, so a failure is traceable to a specific
   step instead of a bare stack trace. If training somehow runs past 30
   minutes (see MAX_TRAIN_MINUTES in the training cell), it stops itself
   cleanly rather than running away, and the save/zip cells after it always
   run regardless of what happened above them.
4. Click **Save Version** (top right), wait for it to finish, then open the
   **Output** tab and download `sl-food-classifier.zip`.
5. Unzip it into `serve/sl-food-classifier/` locally - it's already wired
   into `serve/food_scan.py` as the higher-precision path alongside CLIP.

In [ ]:
!pip install -q -U transformers accelerate datasets scikit-learn

## GPU check (fail fast rather than silently crawl on CPU)

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU detected - this will be extremely slow on CPU. Go to "
        "Settings -> Accelerator -> GPU T4 x2 (or similar) and re-run."
    )

print(f"GPU OK: {torch.cuda.get_device_name(0)}")

## Find the uploaded dataset under /kaggle/input

In [ ]:
from pathlib import Path

# Kaggle nests attached datasets under a path that includes your username
# (e.g. /kaggle/input/datasets/<username>/tripme-food-datasets), not always
# the flat /kaggle/input/tripme-food-datasets/ shape - search for wherever
# the 4 Roboflow export folders actually landed rather than hardcoding one
# specific path shape.
RAW_DATASETS_DIR = None
_kaggle_input = Path("/kaggle/input")
if _kaggle_input.exists():
    # Look for a folder that itself contains subfolders with a COCO
    # annotations file inside (i.e. the parent of the 4 dataset folders).
    for candidate in _kaggle_input.rglob("*"):
        if not candidate.is_dir():
            continue
        subdirs = [p for p in candidate.iterdir() if p.is_dir()]
        if subdirs and any((sd / split / "_annotations.coco.json").exists()
                            for sd in subdirs for split in ("train", "valid", "test")):
            RAW_DATASETS_DIR = candidate
            break

if RAW_DATASETS_DIR is None:
    # Fallback for local/non-Kaggle testing.
    RAW_DATASETS_DIR = Path("data/raw_datasets")

assert RAW_DATASETS_DIR.exists(), (
    f"{RAW_DATASETS_DIR} does not exist - attach the dataset containing the "
    "4 Roboflow COCO export folders via 'Add Input' before running the rest "
    "of this notebook."
)

dataset_dirs = [p for p in RAW_DATASETS_DIR.iterdir() if p.is_dir()]
assert dataset_dirs, f"No dataset folders found under {RAW_DATASETS_DIR}"

print(f"Using: {RAW_DATASETS_DIR}")
print(f"Found {len(dataset_dirs)} dataset folder(s):")
for d in dataset_dirs:
    print(f"  - {d.name}")

## Crop bounding boxes into per-class classification images

Each annotated box becomes one training image (cropped to just that box,
with a small margin) labeled with its class - this converts the
detection-style annotations into a plain image classification dataset.

In [ ]:
import json
import shutil

from PIL import Image

CROPS_DIR = Path("sl_food_crops")
if CROPS_DIR.exists():
    shutil.rmtree(CROPS_DIR)
CROPS_DIR.mkdir()

BOX_MARGIN_FRAC = 0.05  # small margin around each box so the crop isn't razor-tight

# Each of the 4 source datasets has its own root/supercategory placeholder
# class that isn't a real dish - skip these by name rather than treating
# them as classes to train on.
SUPERCATEGORY_NAMES = {
    "food", "lankan", "food detect", "sri lankan food detection",
    "pastryitems-bakeryitems-fooditem",
}


# Class names differ in capitalization/spacing across the source datasets
# even where they mean the same dish (e.g. "Fish Curry" vs "fish_curry",
# "Donut" vs "donut") - normalize before folder naming so they land in one
# folder, not two.
def normalize_class_name(name: str) -> str:
    return name.strip().replace(" ", "_").replace("-", "_").lower()


def resolve_image_path(split_dir: Path, file_name: str) -> Path | None:
    """Roboflow's COCO export has put images directly under the split
    folder in every version we've seen, but export layouts can shift
    between dataset versions - check both that and an `images/` subfolder
    shape rather than assuming one and failing silently on the other."""
    for candidate in (split_dir / file_name, split_dir / "images" / file_name):
        if candidate.is_file():
            return candidate
    return None


def crop_split(split_dir: Path, split_name: str, source_tag: str) -> tuple[int, int]:
    """Returns (crops_written, annotations_skipped). source_tag prefixes
    output filenames so annotation ids from different source datasets can
    never collide with each other."""
    ann_path = split_dir / "_annotations.coco.json"
    if not ann_path.exists():
        print(f"  INFO: no _annotations.coco.json found under {split_dir}")
        return 0, 0
    coco = json.loads(ann_path.read_text(encoding="utf-8"))

    images_by_id = {img["id"]: img for img in coco["images"]}
    categories_by_id = {c["id"]: c["name"] for c in coco["categories"]}

    written, skipped = 0, 0
    for ann in coco["annotations"]:
        img_info = images_by_id.get(ann["image_id"])
        if img_info is None:
            skipped += 1
            continue
        class_name = categories_by_id.get(ann["category_id"], "unknown")
        if normalize_class_name(class_name).replace("_", " ") in SUPERCATEGORY_NAMES or \
           normalize_class_name(class_name) in SUPERCATEGORY_NAMES:
            continue  # root/supercategory placeholder, not a real dish - skip, not an error

        img_path = resolve_image_path(split_dir, img_info["file_name"])
        if img_path is None:
            print(f"  [{source_tag}/{split_name}] ERROR: image file not found for '{img_info['file_name']}' "
                  f"(checked {split_dir / img_info['file_name']} and {split_dir / 'images' / img_info['file_name']})")
            skipped += 1
            continue

        x, y, w, h = ann["bbox"]
        if w <= 0 or h <= 0:
            skipped += 1
            continue
        margin_x, margin_y = w * BOX_MARGIN_FRAC, h * BOX_MARGIN_FRAC
        try:
            with Image.open(img_path) as im:
                left = max(0, x - margin_x)
                top = max(0, y - margin_y)
                right = min(im.width, x + w + margin_x)
                bottom = min(im.height, y + h + margin_y)
                crop = im.convert("RGB").crop((left, top, right, bottom))

                class_dir = CROPS_DIR / split_name / normalize_class_name(class_name)
                class_dir.mkdir(parents=True, exist_ok=True)
                crop.save(class_dir / f"{source_tag}_{ann['id']}.jpg", quality=90)
                written += 1
        except Exception as e:
            print(f"  [{source_tag}/{split_name}] ERROR: skipping annotation {ann['id']} ({img_path.name}): {type(e).__name__}: {e}")
            skipped += 1
    return written, skipped


total_crops = 0
total_skipped = 0
for dataset_idx, dataset_dir in enumerate(dataset_dirs):
    source_tag = f"src{dataset_idx}"
    for split in ("train", "valid", "test"):
        split_dir = dataset_dir / split
        if split_dir.exists():
            n, skipped = crop_split(split_dir, split, source_tag)
            print(f"[{source_tag}={dataset_dir.name}] {split}: {n} crops written, {skipped} annotations skipped")
            total_crops += n
            total_skipped += skipped
        else:
            print(f"[{source_tag}={dataset_dir.name}] {split}: no folder found at {split_dir} (dataset may not have this split)")

print(f"\nTotal crops: {total_crops} (skipped: {total_skipped})")
if total_crops == 0:
    raise SystemExit(
        "No crops were produced at all - check the printout above for which "
        "splits/annotations were skipped and why. Nothing to train on."
    )

class_names = sorted(p.name for p in (CROPS_DIR / "train").iterdir()) if (CROPS_DIR / "train").exists() else []
print(f"Classes found ({len(class_names)}): {class_names}")

## Load crops as a Hugging Face image dataset

In [ ]:
from datasets import load_dataset

# imagefolder infers labels from the subfolder names created above.
ds = load_dataset("imagefolder", data_dir=str(CROPS_DIR))
print(ds)

labels = ds["train"].features["label"].names
print(f"Label list ({len(labels)}): {labels}")

if len(labels) < 2:
    raise SystemExit(
        f"Only {len(labels)} class(es) found - need at least 2 to train a "
        "classifier. Check the crop step above found the expected classes."
    )

## Fine-tune a pretrained ViT on the crops

A `TrainerCallback` below hard-stops training 30 minutes in if it somehow
runs long (e.g. GPU allocation silently failed and it's crawling on CPU) -
guarantees the save/zip cells after it always get a chance to run.

In [ ]:
import time

import numpy as np
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    Trainer,
    TrainingArguments,
    TrainerCallback,
)

BASE_MODEL = "google/vit-base-patch16-224-in21k"
OUTPUT_DIR = "sl-food-classifier"

MAX_TRAIN_MINUTES = 30
run_deadline = time.time() + MAX_TRAIN_MINUTES * 60

processor = AutoImageProcessor.from_pretrained(BASE_MODEL)


def transform(batch):
    images = [img.convert("RGB") for img in batch["image"]]
    inputs = processor(images=images, return_tensors="pt")
    inputs["labels"] = batch["label"]
    return inputs


ds_transformed = ds.with_transform(transform)

model = AutoModelForImageClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(labels),
    id2label={i: l for i, l in enumerate(labels)},
    label2id={l: i for i, l in enumerate(labels)},
    ignore_mismatched_sizes=True,
)


def collate_fn(batch):
    return {
        "pixel_values": torch.stack([item["pixel_values"] for item in batch]),
        "labels": torch.tensor([item["labels"] for item in batch]),
    }


def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    accuracy = (predictions == eval_pred.label_ids).mean()
    return {"accuracy": accuracy}


class TimeLimitCallback(TrainerCallback):
    """Stops training cleanly at a step boundary if it somehow runs past
    MAX_TRAIN_MINUTES (e.g. silently fell back to CPU) - see the comment on
    MAX_TRAIN_MINUTES above for why this exists even though the job is
    normally fast."""

    def on_step_end(self, args, state, control, **kwargs):
        if time.time() >= run_deadline:
            print(f"\nHit the {MAX_TRAIN_MINUTES}-minute safety stop at step {state.global_step} - stopping cleanly to save.")
            control.should_training_stop = True
        return control


eval_split = "validation" if "validation" in ds_transformed else "train"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=15,  # small dataset - more epochs, watch for overfitting via eval accuracy
    learning_rate=3e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=5,
    report_to="none",
    # with_transform() keeps "image" as a lazily-transformed PIL column, not
    # a real tensor - Trainer's default column-pruning (based on the model's
    # forward() signature) drops it before compute_metrics/collate_fn ever
    # see it, causing a silent KeyError('image') during eval that (via the
    # try/except around trainer.train()) can look like training "succeeded"
    # while actually never learning anything. Keep all columns instead.
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_transformed["train"],
    eval_dataset=ds_transformed[eval_split],
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    callbacks=[TimeLimitCallback()],
)

train_result = trainer.train()
print(f"\nTraining finished normally after {train_result.metrics.get('epoch', '?')} epochs.")

## Evaluate and save

Runs even if the cell above hit an error or the time-limit stop, so
whatever was learned so far still gets written to disk.

In [ ]:
try:
    eval_results = trainer.evaluate()
    print("Final eval results:", eval_results)
except Exception as e:
    print(f"Evaluation failed ({e}) - saving the model anyway.")

print(
    "\nWith only ~1000-1300 training images across 20+ classes, treat any "
    "accuracy number above as a rough signal, not a production benchmark."
)

trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

## Smoke test on a few validation images

In [ ]:
from transformers import pipeline

try:
    clf = pipeline("image-classification", model=OUTPUT_DIR, image_processor=OUTPUT_DIR)
    sample_split = ds[eval_split] if eval_split in ds else ds["train"]
    for i in range(min(5, len(sample_split))):
        example = sample_split[i]
        true_label = labels[example["label"]]
        preds = clf(example["image"])
        top = preds[0]
        print(f"True: {true_label:20s} | Predicted: {top['label']:20s} (score={top['score']:.2f})")
except Exception as e:
    print(f"Smoke test failed ({e}) - not fatal, the saved model files are still valid to download.")

## Zip the fine-tuned model for download

Kaggle has no direct browser-auto-download API (unlike Colab's
files.download()) - the only way to get files out is: zip them here,
click **Save Version** (top right) so Kaggle persists /kaggle/working as a
Version Output, then download the zip from the notebook's **Output** tab.
Runs regardless of whether training/eval/smoke-test above hit errors -
whatever got saved to OUTPUT_DIR is worth getting off this session.

In [ ]:
OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")

# OUTPUT_DIR also contains checkpoint-XXX/ subfolders written by the
# Trainer's save_strategy="epoch" (full optimizer state etc, ~600-700MB
# each even with save_total_limit=2) - trainer.save_model()/processor
# .save_pretrained() above already wrote the actual final model we want
# directly into OUTPUT_DIR, so drop the checkpoint subfolders before
# zipping rather than shipping ~2GB of training-resume state nobody needs
# for inference.
for ckpt_dir in Path(OUTPUT_DIR).glob("checkpoint-*"):
    shutil.rmtree(ckpt_dir)
    print(f"Removed {ckpt_dir} (training checkpoint, not needed for inference)")

zip_path = shutil.make_archive(str(OUTPUT_ROOT / "sl-food-classifier"), "zip", OUTPUT_DIR)
size_mb = Path(zip_path).stat().st_size / (1024 * 1024)
print(f"Zipped to {zip_path} ({size_mb:.0f} MB)")

## Upload the zip straight to Google Drive (avoids the 2GB browser download)

Same idea as Colab's `drive.mount()`, but since Kaggle can't mount Drive
directly, this uses OAuth: running the cell below prints a Google sign-in
link, you click it, sign in, allow access, and paste the code it gives you
back into the prompt. After that it uploads `sl-food-classifier.zip`
straight to your Drive - no local browser download needed at all.

In [ ]:
!pip install -q -U pydrive2

from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive

# Kaggle has no browser of its own, so this can't pop up a sign-in window
# like it would on your own machine - it prints a link instead. Click it,
# sign in with the Google account whose Drive you want the file in, allow
# access, then paste the code it shows you into the input box that appears
# below this cell.
gauth = GoogleAuth()
gauth.settings["get_refresh_token"] = True
gauth.CommandLineAuth()

drive = GoogleDrive(gauth)

upload_file = drive.CreateFile({"title": "sl-food-classifier.zip"})
upload_file.SetContentFile(zip_path)
upload_file.Upload()

print(f"\nUploaded to Google Drive as '{upload_file['title']}'")
print(f"File ID: {upload_file['id']}")
print("Check 'My Drive' (root folder) in Google Drive - it'll be there, "
      "no local download needed.")